# 02 Multitask Model

Define the tokenizer, dataset objects, dataloaders, and the shared-encoder multi-task model.

## Workflow

- Load processed ABSA and emotion data
- Build label dictionaries
- Initialize tokenizer
- Define ABSA and emotion dataset classes
- Build dataloaders
- Define shared BERT encoder with two task heads
- Run a forward-pass sanity check

In [1]:
from pathlib import Path
import json
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

pd.set_option('display.max_colwidth', 120)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
MODEL_DIR = PROJECT_DIR / 'models'

DEVICE, MODEL_DIR

d:\important\Conda_Envs\base_D\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


(device(type='cuda'), WindowsPath('d:/15_MAI/pynotebook/nlpProject/models'))

In [2]:
absa_df = pd.read_csv(PROCESSED_DIR / 'absa_semeval_2014.csv')
emotion_df = pd.read_csv(PROCESSED_DIR / 'emotion_goemotions_6class.csv')

with open(PROCESSED_DIR / 'label_metadata.json', 'r', encoding='utf-8') as f:
    label_metadata = json.load(f)

ABSA_LABEL2ID = label_metadata['absa_label2id']
EMOTION_LABEL2ID = label_metadata['emotion_label2id']
ID2ABSA_LABEL = {int(v): k for k, v in ABSA_LABEL2ID.items()}
ID2EMOTION_LABEL = {int(v): k for k, v in EMOTION_LABEL2ID.items()}

absa_df.head(), emotion_df.head()

(   task  domain  split  sample_id  \
 0  absa  laptop  train       1964   
 1  absa  laptop  train        462   
 2  absa  laptop  train        908   
 3  absa  laptop  train       1895   
 4  absa  laptop  train        963   
 
                                                                                                                       text  \
 0                                                                    This computer was so challenging to carry and handle.   
 1                                                                                 Unable to boot up this brand new laptop.   
 2  Also, my sister got the exact same laptop (since they were so cheap) and after 8 months, the screen split in half ju...   
 3  Overall I feel this netbook was poor quality, had poor performance, although it did have great battery life when it ...   
 4                                    Another THREE weeks later I had my laptop back with a new mousepad, keys, and casing.   
 
      

In [3]:
absa_train = absa_df[absa_df['split'] == 'train'].reset_index(drop=True)
absa_dev = absa_df[absa_df['split'] == 'dev'].reset_index(drop=True)
absa_test = absa_df[absa_df['split'] == 'test'].reset_index(drop=True)

emotion_train = emotion_df[emotion_df['split'] == 'train'].reset_index(drop=True)
emotion_dev = emotion_df[emotion_df['split'] == 'dev'].reset_index(drop=True)
emotion_test = emotion_df[emotion_df['split'] == 'test'].reset_index(drop=True)

print('ABSA train/dev/test:', len(absa_train), len(absa_dev), len(absa_test))
print('Emotion train/dev/test:', len(emotion_train), len(emotion_dev), len(emotion_test))

ABSA train/dev/test: 4840 605 606
Emotion train/dev/test: 39555 4946 4968


## Tokenizer

In [4]:
MODEL_NAME = str(MODEL_DIR)
MAX_LENGTH = 128
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer

BertTokenizer(name_or_path='d:\15_MAI\pynotebook\nlpProject\models', vocab_size=30522, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

## Dataset Classes

In [5]:
class ABSADataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int = 128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = self.tokenizer(
            text=str(row['text']),
            text_pair=str(row['aspect']),
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(int(row['label_id']), dtype=torch.long),
            'task': 'absa',
            'text': row['text'],
            'aspect': row['aspect'],
        }

        if 'token_type_ids' in encoded:
            item['token_type_ids'] = encoded['token_type_ids'].squeeze(0)

        return item

class EmotionDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int = 128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = self.tokenizer(
            text=str(row['text']),
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(int(row['label_id']), dtype=torch.long),
            'task': 'emotion',
            'text': row['text'],
        }

        if 'token_type_ids' in encoded:
            item['token_type_ids'] = encoded['token_type_ids'].squeeze(0)

        return item

In [6]:
absa_train_dataset = ABSADataset(absa_train, tokenizer, MAX_LENGTH)
emotion_train_dataset = EmotionDataset(emotion_train, tokenizer, MAX_LENGTH)

absa_train_dataset[0]

{'input_ids': tensor([  101,  2023,  3274,  2001,  2061, 10368,  2000,  4287,  1998,  5047,
          1012,   102,  4287,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

## DataLoaders

In [7]:
absa_train_loader = DataLoader(ABSADataset(absa_train, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
absa_dev_loader = DataLoader(ABSADataset(absa_dev, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
absa_test_loader = DataLoader(ABSADataset(absa_test, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

emotion_train_loader = DataLoader(EmotionDataset(emotion_train, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
emotion_dev_loader = DataLoader(EmotionDataset(emotion_dev, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
emotion_test_loader = DataLoader(EmotionDataset(emotion_test, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

len(absa_train_loader), len(emotion_train_loader)

(303, 2473)

## Multi-task Model

In [8]:
class MultiTaskBERT(nn.Module):
    def __init__(self, model_name, absa_num_labels=4, emotion_num_labels=6, dropout=0.1):
        super().__init__()
        self.encoder = BertModel.from_pretrained(model_name, local_files_only=True)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.absa_classifier = nn.Linear(hidden_size, absa_num_labels)
        self.emotion_classifier = nn.Linear(hidden_size, emotion_num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, task='absa'):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        cls_output = outputs.last_hidden_state[:, 0]
        cls_output = self.dropout(cls_output)

        if task == 'absa':
            logits = self.absa_classifier(cls_output)
        elif task == 'emotion':
            logits = self.emotion_classifier(cls_output)
        else:
            raise ValueError("task must be 'absa' or 'emotion'")

        return logits

In [9]:
model = MultiTaskBERT(
    model_name=MODEL_NAME,
    absa_num_labels=len(ABSA_LABEL2ID),
    emotion_num_labels=len(EMOTION_LABEL2ID)
).to(DEVICE)

model

MultiTaskBERT(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementw

## Sanity Check

In [10]:
absa_batch = next(iter(absa_train_loader))

absa_input_ids = absa_batch['input_ids'].to(DEVICE)
absa_attention_mask = absa_batch['attention_mask'].to(DEVICE)
absa_token_type_ids = absa_batch.get('token_type_ids')
if absa_token_type_ids is not None:
    absa_token_type_ids = absa_token_type_ids.to(DEVICE)

absa_logits = model(
    input_ids=absa_input_ids,
    attention_mask=absa_attention_mask,
    token_type_ids=absa_token_type_ids,
    task='absa'
)

print('ABSA logits shape:', absa_logits.shape)

ABSA logits shape: torch.Size([16, 4])


In [11]:
emotion_batch = next(iter(emotion_train_loader))

emotion_input_ids = emotion_batch['input_ids'].to(DEVICE)
emotion_attention_mask = emotion_batch['attention_mask'].to(DEVICE)
emotion_token_type_ids = emotion_batch.get('token_type_ids')
if emotion_token_type_ids is not None:
    emotion_token_type_ids = emotion_token_type_ids.to(DEVICE)

emotion_logits = model(
    input_ids=emotion_input_ids,
    attention_mask=emotion_attention_mask,
    token_type_ids=emotion_token_type_ids,
    task='emotion'
)

print('Emotion logits shape:', emotion_logits.shape)

Emotion logits shape: torch.Size([16, 6])
